# OpenTools evaluation, contribution, and MCP demo

This notebook exercises the implemented paths with real local code. It does not mock tool, MCP, evaluation, or LLM results. Optional integrations are explicitly reported as `SKIPPED` when their packages are unavailable.

From the repository root, install the full demo environment if needed:

```bash
python -m pip install -e '.[mcp]'
```

In [1]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'src/opentools').is_dir()), None)
if REPO is None:
    raise RuntimeError('Run this notebook from within the OpenTools repository.')
sys.path.insert(0, str(REPO / 'src'))
print(f'Repository: {REPO}')


Repository: /Users/hydang/workspace/misc/opentools/opentools


## 1. Static risk inspection

Static inspection parses source without importing or executing the tool. Its findings are review signals, not a security guarantee.

In [2]:
from opentools.evaluation import inspect_source

calculator_source = REPO / 'src/opentools/tools/calculator/tool.py'
inspection = inspect_source(calculator_source)
print({
    'risk_level': inspection['risk_level'],
    'files_scanned': inspection['files_scanned'],
    'observed_credentials': inspection['observed_credentials'],
    'finding_kinds': sorted({item['kind'] for item in inspection['findings']}),
})
assert inspection['files_scanned'] == 1


{'risk_level': 'low', 'files_scanned': 1, 'observed_credentials': [], 'finding_kinds': ['evaluation_artifact_write']}


## 2. Build the evaluation inventory in memory

This reads real tool sources and existing result artifacts without executing every tool. The scheduled CI workflow uses the related refresh command to propose table changes through a pull request.

In [3]:
from collections import Counter
from opentools.inventory import build_index

tools_root = REPO / 'src/opentools/tools'
inventory = build_index(tools_root)
statuses = Counter(record['evaluation_status'] for record in inventory['tools'].values())
print(f"Discovered tools: {len(inventory['tools'])}")
print(f'Evaluation statuses: {dict(statuses)}')
print('Calculator:', inventory['tools']['Calculator_Tool'])


Discovered tools: 42
Evaluation statuses: {'not_evaluated': 13, 'historical': 29}
Calculator: {'folder': 'calculator', 'class_name': 'Calculator_Tool', 'risk_level': 'low', 'evaluation_status': 'historical', 'freshness': 'unknown', 'evaluated': True, 'average_accuracy': 100.0, 'final_accuracy': {'run_1': 100.0, 'run_2': 100.0, 'run_3': 100.0}, 'total_questions': 100, 'last_evaluated': None, 'result_source': 'src/opentools/tools/calculator/test_result.json'}


## 3. Convert and execute a real submitted function

Conversion itself does not execute the submission and marks functional evaluation as `not_run`. This cell then deliberately imports the generated bundle and calls it, demonstrating the generated wrapper with an actual result.

In [4]:
import importlib.util
import inspect
import tempfile
from opentools.conversion import convert_submission
from opentools.core.base import BaseTool

with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    source = root / 'submitted.py'
    readme = root / 'README.md'
    source.write_text(
        'def multiply(left: int, right: int) -> int:\n'
        '    \"\"\"Multiply two integers.\"\"\"\n'
        '    return left * right\n',
        encoding='utf-8',
    )
    readme.write_text('# Local multiplier\n', encoding='utf-8')
    converted = convert_submission(
        source, readme, root / 'contributions',
        name='Local Multiplier', license_name='Apache-2.0',
    )
    print('Conversion:', converted['manifest']['conversion_status'])
    print('Evaluation:', converted['manifest']['functional_evaluation'])
    print('Risk:', converted['manifest']['risk']['risk_level'])

    tool_file = Path(converted['bundle']) / 'tool.py'
    spec = importlib.util.spec_from_file_location('local_multiplier_bundle', tool_file)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    tool_class = next(
        value for _, value in inspect.getmembers(module, inspect.isclass)
        if value is not BaseTool and issubclass(value, BaseTool)
    )
    observed = tool_class().run(left=6, right=7)
    print('Observed invocation:', observed)
    assert observed == {'result': 42, 'success': True}


Conversion: converted
Evaluation: {'status': 'not_run', 'reason': 'Submitted code is not executed during conversion.'}
Risk: low
Observed invocation: {'result': 42, 'success': True}


## 4. Invoke Calculator through the real MCP registration

The MCP server is default-deny. The cell temporarily enables and allowlists only `Calculator_Tool`, calls the actual registered MCP operation, and restores the environment afterward. It skips explicitly if the MCP SDK is not installed.

In [5]:
import asyncio
import importlib.util
import os

if importlib.util.find_spec('mcp') is None:
    print("SKIPPED: install the MCP SDK with `python -m pip install -e '.[mcp]'`.")
else:
    from opentools.mcp_server import create_server

    keys = [
        'OPENTOOLS_MCP_ALLOWED_TOOLS',
        'OPENTOOLS_MCP_ALLOW_TOOL_CALLS',
        'OPENTOOLS_MCP_MAX_RISK',
    ]
    previous = {key: os.environ.get(key) for key in keys}
    try:
        os.environ['OPENTOOLS_MCP_ALLOWED_TOOLS'] = 'Calculator_Tool'
        os.environ['OPENTOOLS_MCP_ALLOW_TOOL_CALLS'] = '1'
        os.environ['OPENTOOLS_MCP_MAX_RISK'] = 'low'
        server = create_server()
        registered = server._tool_manager._tools
        print('Registered MCP operations:', sorted(registered))
        observed = await registered['call_opentool'].fn(
            'Calculator_Tool',
            {'operation': 'add', 'values': [1, 2, 3]},
        )
        print('Observed MCP result:', observed)
        assert observed['status'] == 'completed'
        assert observed['result']['result'] == '6'
    finally:
        for key, value in previous.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value


Registered MCP operations: ['call_opentool', 'evaluate_opentool', 'inspect_opentool', 'list_opentools']
Observed MCP result: {'status': 'completed', 'tool_name': 'Calculator_Tool', 'result': {'result': '6', 'success': True}}


To test the transport from another MCP application, start the server separately:

```bash
OPENTOOLS_MCP_ALLOWED_TOOLS=Calculator_Tool \
OPENTOOLS_MCP_ALLOW_TOOL_CALLS=1 \
opentools-mcp --transport stdio
```

For local Streamable HTTP, use `opentools-mcp --transport streamable-http --host 127.0.0.1 --port 8000`.

[07/07/26 10:38:52] INFO     Received session ID: f2a2b5df1ebf4b4e8b15277fd9af909c           ]8;id=885746;file:///opt/miniconda3/envs/opentools/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=655758;file:///opt/miniconda3/envs/opentools/lib/python3.12/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=751962;file:///opt/miniconda3/envs/opentools/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=75753;file:///opt/miniconda3/envs/opentools/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

MCP tools: ['list_opentools', 'inspect_opentool', 'evaluate_opentool', 'call_opentool']
Inspection: {'result': {'tool_card': {'name': 'Calculator_Tool', 'version': '1.0.0', 'description': "A comprehensive mathematical calculation operations tool that provides basic arithmetic, logarithms, combinatorics, trigonometry, constants parsing,  and common unit conversions with precise decimal handling. CAPABILITIES: Basic arithmetic operations (add, subtract, multiply, divide), power and root calculations  (power, sqrt, exp), logarithmic functions (log base-10, ln natural log), combinatorial mathematics (combinations, permutations, factorial), number theory  (GCD, LCM, remainder), trigonometric and hyperbolic functions (sin, cos, tan, asin, acos, atan, sinh, cosh, tanh), constant-aware parsing (pi, e, tau, c, h, hbar,  k, na, g0, R) and concise unit conversions (length, mass, time, temperature, pressure, energy). SYNONYMS: math calculator, mathematical operations, arithmetic tool,  scientific 

## 5. Existing contribution WebUI

Tool contribution is integrated into the existing OpenTools Hugging Face Space rather than a second local WebUI. The Space's **Contribute Tools** tab accepts README + `tool.py`, performs non-executing review, and creates a pending-maintainer-review bundle: https://huggingface.co/spaces/opentools/opentools

In [ ]:
print('Contribution UI: https://huggingface.co/spaces/opentools/opentools')
print('Use the Contribute Tools tab; uploaded code is not executed by the Space.')


## 6. Opt-in real LLM judge

The judge is intentionally disabled by default because it may require credentials and incur provider cost. Set `RUN_LLM_JUDGE=1` and configure the selected model provider before running this cell. A missing credential or provider error is reported as a real failure; no fallback verdict is generated.

In [10]:
from opentools.evaluation import build_tool_card, judge_evaluation_report
from opentools.inventory import _load_tool, discover_tool_sources

if os.getenv('RUN_LLM_JUDGE') != '1':
    print('NOT RUN: set RUN_LLM_JUDGE=1 and real provider credentials to request a judge verdict.')
else:
    discovered = discover_tool_sources(tools_root)
    item = discovered['Calculator_Tool']
    tool = _load_tool('Calculator_Tool', item['source'])
    static = inspect_source(item['source'])
    report = {
        'inspection': static,
        'tool_card': build_tool_card(tool, static, {'status': 'not_run'}),
    }
    model = os.getenv('OPENTOOLS_JUDGE_MODEL', 'gpt-4o-mini')
    verdict = judge_evaluation_report(report, model=model)
    print(verdict)


{'status': 'completed', 'model': 'gpt-4o-mini', 'advisory_only': True, 'can_override_preflight': False, 'preflight_blocked': False, 'eligible_for_automatic_acceptance': False, 'judgment': {'recommendation': 'approve', 'scores': {'documentation': 4, 'test_evidence': 5, 'output_contract': 4, 'maintainability': 4}, 'concerns': [], 'required_actions': [], 'rationale': 'The tool has comprehensive documentation, strong test evidence with consistent accuracy, and a clear output contract. No significant concerns were identified.'}}


## Recurring evaluation commands

To run a real low-risk evaluation and update the generated inventory locally:

```bash
opentools evaluate-all --tools Calculator_Tool --max-risk low --discard-raw-results
```

The repository's weekly GitHub workflow performs the configured refresh on an automation branch and opens or updates a pull request instead of writing directly to `main`.